# Ekspor Tinjauan & Lembar Persetujuan Master Data Aset
### Kategori & Subkategori: Migrasi Data AMS Legacy ke Microsoft SQL Server Smart

---

## 1. Latar Belakang & Tujuan
Master data **Kategori** dan **Subkategori** adalah fondasi relasional utama dari seluruh data aset fisik. Konsensus dan persetujuan tertulis (*approval*) dari **Tim Pengelola Aset** dan **Tim IT** sangat krusial sebelum migrasi ribuan unit aset dijalankan.

Notebook ini bertugas secara otomatis mengolah data staging/final hasil pembersihan dan mengekspor:
1. **Workbook Excel Siap-Tinjau** (`data/smart/review/Review_Master_Kategori_dan_Subkategori.xlsx`):
   - Menggunakan tata letak profesional, navigasi beku (*freeze panes*), indikator status warna, dan dropdown persetujuan.
   - Terdiri dari 5 lembar kerja: `Ringkasan_Eksekutif`, `Master_Kategori`, `Master_Subkategori`, `Keputusan_Pembersihan_Data`, dan `Pemetaan_AMS_ke_Smart`.
2. **Dokumen Ringkasan Eksekutif Markdown** (`data/smart/review/ringkasan_review_kategori.md`):
   - Dapat dibaca langsung di repositori Git tanpa memerlukan aplikasi spreadsheet.


In [1]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from pathlib import Path
from datetime import datetime

# Dynamic Path Configuration
def get_data_dir():
    candidates = [
        Path("data"),
        Path("../data"),
        Path("/home/jovyan/data"),
        Path("/home/ran/Devs/ams-smart-migration/data"),
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists():
            return p.resolve()
    return Path("../data").resolve()

DATA_DIR = get_data_dir()
RAW_PATH = DATA_DIR / "ams" / "tb_assets_category_202609291319.csv"
CAT_FINAL_PATH = DATA_DIR / "smart" / "final" / "category" / "categories.csv"
SUB_FINAL_PATH = DATA_DIR / "smart" / "final" / "subcategory" / "subcategories.csv"
CAT_MAP_PATH = DATA_DIR / "smart" / "staging" / "category" / "category_ams_mapping.csv"
SUB_MAP_PATH = DATA_DIR / "smart" / "staging" / "subcategory" / "subcategory_ams_mapping.csv"
RELEG_PATH = DATA_DIR / "smart" / "staging" / "category" / "relegated_subcategories.csv"

OUT_DIR = DATA_DIR / "smart" / "review"
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = OUT_DIR / "Review_Master_Kategori_dan_Subkategori.xlsx"
MD_PATH = OUT_DIR / "ringkasan_review_kategori.md"

print(f"Data Dir: {DATA_DIR}")
print(f"Output Excel: {EXCEL_PATH}")

# Load datasets
raw = pd.read_csv(RAW_PATH)
cat_final = pd.read_csv(CAT_FINAL_PATH)
sub_final = pd.read_csv(SUB_FINAL_PATH)
cat_map = pd.read_csv(CAT_MAP_PATH)
sub_map = pd.read_csv(SUB_MAP_PATH)
releg = pd.read_csv(RELEG_PATH)

print(f"Loaded: Raw={len(raw)}, Categories={len(cat_final)}, Subcategories={len(sub_final)}")

# Setup OpenPyXL Workbook
wb = openpyxl.Workbook()
wb.remove(wb.active)

# Fonts & Styles
font_title = Font(name="Calibri", size=16, bold=True, color="1F4E78")
font_subtitle = Font(name="Calibri", size=11, italic=True, color="595959")
font_section = Font(name="Calibri", size=13, bold=True, color="1F4E78")
font_header = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
font_header_feedback = Font(name="Calibri", size=11, bold=True, color="000000")
font_data = Font(name="Calibri", size=10)
font_bold = Font(name="Calibri", size=10, bold=True)
font_italic = Font(name="Calibri", size=10, italic=True)

# Fills
fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
fill_header_sub = PatternFill(start_color="2F5597", end_color="2F5597", fill_type="solid")
fill_header_yellow = PatternFill(start_color="FFD966", end_color="FFD966", fill_type="solid")
fill_header_green = PatternFill(start_color="C6E0B4", end_color="C6E0B4", fill_type="solid")
fill_zebra = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
fill_white = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
fill_special = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

# Borders
thin_gray = Side(style="thin", color="D9D9D9")
border_cell = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)
border_header = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=Side(style="medium", color="1F4E78"))

# Alignments
align_left = Alignment(horizontal="left", vertical="center")
align_center = Alignment(horizontal="center", vertical="center")
align_right = Alignment(horizontal="right", vertical="center")
align_wrap_left = Alignment(horizontal="left", vertical="center", wrap_text=True)

# -------------------------------------------------------------------------
# Sheet 1: Ringkasan_Eksekutif
# -------------------------------------------------------------------------
ws1 = wb.create_sheet(title="Ringkasan_Eksekutif")
ws1.views.sheetView[0].showGridLines = True

ws1.cell(row=2, column=2, value="TINJAUAN & PERSETUJUAN MASTER DATA KATEGORI & SUBKATEGORI").font = font_title
ws1.cell(row=3, column=2, value="Migrasi Data Sistem AMS Legacy ke Sistem Manajemen Aset Smart").font = font_subtitle
ws1.cell(row=4, column=2, value=f"Tanggal Dibuat: {datetime.now().strftime('%d %B %Y')} | Status: Menunggu Tinjauan Tim Aset & IT").font = font_italic

ws1.cell(row=6, column=2, value="1. Latar Belakang & Tujuan").font = font_section
desc_text = (
    "Master data Kategori dan Subkategori merupakan fondasi relasional paling kritis dalam pengelolaan aset. "
    "Sebelum proses migrasi dan pembersihan data ribuan unit aset fisik dijalankan, struktur master data harus memiliki "
    "standar penamaan baku, kejelasan klasifikasi, serta konsensus dan persetujuan tertulis dari Tim Pengelola Aset dan Tim IT.\n\n"
    "Dokumen ini memfasilitasi verifikasi, penyesuaian, serta persetujuan terhadap keputusan transformasi dari sistem legacy AMS ke skema database Smart."
)
c_desc = ws1.cell(row=7, column=2, value=desc_text)
c_desc.font = font_data
c_desc.alignment = align_wrap_left
ws1.merge_cells(start_row=7, start_column=2, end_row=9, end_column=8)

ws1.cell(row=11, column=2, value="2. Ringkasan Metrik Transformasi Data").font = font_section
headers_metric = ["Metrik Indikator", "Data AMS Legacy", "Data Target Smart", "Keterangan & Rationale"]
for col_idx, text in enumerate(headers_metric, start=2):
    cell = ws1.cell(row=12, column=col_idx, value=text)
    cell.font = font_header
    cell.fill = fill_header
    cell.alignment = align_center
    cell.border = border_header

metrics_rows = [
    ("Jumlah Kategori Induk", "15 Baris (control='F')", "13 Kategori Aktif", "Penggabungan kategori duplikat (PROP & KEND) dan eksklusi kategori nonaktif (PUR)"),
    ("Jumlah Subkategori", "113 Baris (control='T')", "110 Subkategori Standar", "Deduplikasi subkategori LIFT & MOBIL, pembentukan kode terstruktur <INDUK>-<ANAK>"),
    ("Subkategori Yatim (Orphan)", "24 Subkategori tanpa Parent", "0 Subkategori Yatim", "24 subkategori dihubungkan kembali ke kategori induk aslinya berdasarkan prefix kode"),
    ("Kategori Baru Terstandar", "Belum ada pemisahan", "1 Kategori Baru (TLKM)", "Pemisahan perangkat telekomunikasi (Telepon, HT, Fax, HP) ke kategori khusus"),
    ("Resolusi Bentrok Kode & Reklasifikasi", "1 Bentrok Kode (WH pada ACC)", "Terselesaikan (WB & WTH)", "Whiteboard tetap di ACC (ACC-WB), Water Heater direklasifikasi ke Mesin (MES-WTH)")
]

for row_idx, rdata in enumerate(metrics_rows, start=13):
    f_zebra = fill_zebra if row_idx % 2 == 1 else fill_white
    for col_idx, val in enumerate(rdata, start=2):
        cell = ws1.cell(row=row_idx, column=col_idx, value=val)
        cell.font = font_bold if col_idx in [2, 4] else font_data
        cell.fill = f_zebra
        cell.border = border_cell
        cell.alignment = align_center if col_idx in [3, 4] else align_left

ws1.cell(row=20, column=2, value="3. Panduan Pengisian Feedback untuk Tim Aset & Tim IT").font = font_section
guide_text = (
    "1. Buka sheet 'Master_Kategori' dan 'Master_Subkategori'.\n"
    "2. Kolom berwarna HIJAU ('Status Persetujuan') dan KUNING ('Catatan & Masukan Tim') disiapkan khusus untuk masukan Anda.\n"
    "3. Pilih Status: 'Disetujui', 'Perlu Penyesuaian', atau 'Ditolak'.\n"
    "4. Apabila memilih 'Perlu Penyesuaian' atau 'Ditolak', mohon sertakan rekomendasi perbaikan pada kolom 'Catatan & Masukan Tim'.\n"
    "5. Periksa sheet 'Keputusan_Pembersihan_Data' untuk melihat pertimbangan teknis di balik penggabungan dan perubahan kode."
)
c_guide = ws1.cell(row=21, column=2, value=guide_text)
c_guide.font = font_data
c_guide.alignment = align_wrap_left
ws1.merge_cells(start_row=21, start_column=2, end_row=26, end_column=8)

# -------------------------------------------------------------------------
# Sheet 2: Master_Kategori
# -------------------------------------------------------------------------
ws2 = wb.create_sheet(title="Master_Kategori")
ws2.views.sheetView[0].showGridLines = True
ws2.freeze_panes = "A2"

headers_cat = [
    "ID Smart", "Kode Kategori", "Nama Kategori", "Kode AMS Terkait", 
    "Jumlah Subkategori", "Catatan Normalisasi", "Status Persetujuan", "Catatan & Masukan Tim Aset & IT"
]

for col_idx, h in enumerate(headers_cat, start=1):
    cell = ws2.cell(row=1, column=col_idx, value=h)
    cell.border = border_header
    if col_idx == 7:
        cell.font = font_header_feedback
        cell.fill = fill_header_green
        cell.alignment = align_center
    elif col_idx == 8:
        cell.font = font_header_feedback
        cell.fill = fill_header_yellow
        cell.alignment = align_center
    else:
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = align_center

cat_ams_dict = cat_map.groupby("category_id")["ams_code"].apply(lambda s: ", ".join(s.astype(str))).to_dict()
sub_count_dict = sub_final.groupby("category_id").size().to_dict()

cat_notes = {
    1: "Perabotan kantor (kursi, meja, lemari, rak, sofa).",
    2: "Aksesoris & fixture kantor (whiteboard, jam dinding, bracket, brankas).",
    3: "Komputer desktop, notebook, workstation, server IT.",
    4: "Monitor komputer (tabung, LCD/LED display).",
    5: "Perangkat periferal, storage eksternal, multimedia, networking.",
    6: "Printer inkjet, laserjet, plotter, ID card printer.",
    7: "Lisensi perangkat lunak engineering dan aplikasi perkantoran.",
    8: "Penggabungan kategori Properti (170000) dan Gedung (220000).",
    9: "Mesin dan utilitas gedung (water heater, genset, lift, panel listrik, pompa air).",
    10: "Penggabungan kategori Kendaraan operasional (200000) dan Otomotif (250000).",
    11: "Peralatan elektronik umum, keamanan gedung, sensor, dan sound system (CCTV, TV).",
    12: "Kategori khusus sistem tata udara dan pendingin ruangan (AC Split, AC Portable, AHU).",
    13: "Kategori mandiri baru untuk perangkat telekomunikasi (Telepon, Smartphone, HT, Fax)."
}

for row_idx, r in cat_final.iterrows():
    c_id = int(r["id"])
    c_code = r["code"]
    c_name = r["name"]
    ams_codes = cat_ams_dict.get(c_id, "-")
    if c_id == 13:
        ams_codes = "110004, 140007, 240013, 240015, 110030"
    num_subs = sub_count_dict.get(c_id, 0)
    note = cat_notes.get(c_id, "")
    
    row_vals = [c_id, c_code, c_name, ams_codes, num_subs, note, "", ""]
    r_excel = row_idx + 2
    f_zebra = fill_zebra if r_excel % 2 == 1 else fill_white
    
    for c_idx, val in enumerate(row_vals, start=1):
        c = ws2.cell(row=r_excel, column=c_idx, value=val)
        c.font = font_data
        c.border = border_cell
        if c_idx == 7:
            c.alignment = align_center
            c.fill = fill_white
        elif c_idx == 8:
            c.alignment = align_left
            c.fill = fill_white
        elif c_idx in [1, 2, 4, 5]:
            c.alignment = align_center
            c.fill = f_zebra
            if c_idx in [1, 2]:
                c.font = font_bold
        else:
            c.alignment = align_left
            c.fill = f_zebra

# -------------------------------------------------------------------------
# Sheet 3: Master_Subkategori
# -------------------------------------------------------------------------
ws3 = wb.create_sheet(title="Master_Subkategori")
ws3.views.sheetView[0].showGridLines = True
ws3.freeze_panes = "A2"

headers_sub = [
    "ID Smart", "Kode Kategori", "Nama Kategori", "Kode Subkategori", "Nama Subkategori",
    "Deskripsi", "Barang Habis Pakai?", "Kode AMS Terkait", "Nama AMS Asal",
    "Status Penanganan Khusus", "Status Persetujuan", "Catatan & Masukan Tim Aset & IT"
]

for col_idx, h in enumerate(headers_sub, start=1):
    cell = ws3.cell(row=1, column=col_idx, value=h)
    cell.border = border_header
    if col_idx == 11:
        cell.font = font_header_feedback
        cell.fill = fill_header_green
        cell.alignment = align_center
    elif col_idx == 12:
        cell.font = font_header_feedback
        cell.fill = fill_header_yellow
        cell.alignment = align_center
    else:
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = align_center

sub_ams_grouped = sub_map.groupby("subcategory_id").agg({
    "ams_code": lambda s: ", ".join(s.astype(str)),
    "ams_id": list
}).to_dict(orient="index")

raw_dict = raw.set_index("code")["category"].to_dict()
releg_set = set(releg["code"].astype(str).tolist())
cat_info = cat_final.set_index("id").to_dict(orient="index")

for row_idx, r in sub_final.iterrows():
    s_id = int(r["id"])
    s_code = r["code"]
    s_name = r["name"]
    s_desc = r["description"] if pd.notna(r["description"]) else "-"
    is_con = "Ya" if r["is_consumable"] else "Tidak"
    c_id = int(r["category_id"])
    c_code = cat_info[c_id]["code"]
    c_name = cat_info[c_id]["name"]
    
    ams_entry = sub_ams_grouped.get(s_id, {"ams_code": "-", "ams_id": []})
    ams_codes_str = ams_entry["ams_code"]
    
    raw_cats = []
    is_relegated = False
    for ac in str(ams_codes_str).split(", "):
        if ac in raw_dict:
            raw_cats.append(str(raw_dict[ac]))
        if ac in releg_set:
            is_relegated = True
    ams_raw_name = ", ".join(raw_cats) if raw_cats else "-"
    
    flag = "Standar"
    if s_code == "ACC-WB":
        flag = "Standardisasi Kode Bentrok (ACC-WB dari kode WH)"
    elif s_code == "MES-WTH":
        flag = "Reklasifikasi ke Mesin & Standardisasi Kode (MES-WTH dari WH)"
    elif s_code == "MES-LIF":
        flag = "Deduplikasi Item (Penggabungan Lift Mesin & Elektronik)"
    elif s_code == "KEND-MO":
        flag = "Deduplikasi Item (Penggabungan Mobil & Kendaraan Ops)"
    elif is_relegated:
        flag = "Subkategori Yatim AMS Ditautkan Ulang ke Induk"
    elif c_id == 13:
        flag = "Reklasifikasi ke Kategori Baru Telekomunikasi"
        
    r_excel = row_idx + 2
    f_zebra = fill_zebra if r_excel % 2 == 1 else fill_white
    f_current = fill_special if flag != "Standar" else f_zebra
    
    row_vals = [
        s_id, c_code, c_name, s_code, s_name, s_desc, is_con,
        ams_codes_str, ams_raw_name, flag, "", ""
    ]
    
    for c_idx, val in enumerate(row_vals, start=1):
        c = ws3.cell(row=r_excel, column=c_idx, value=val)
        c.font = font_data
        c.border = border_cell
        if c_idx == 11:
            c.alignment = align_center
            c.fill = fill_white
        elif c_idx == 12:
            c.alignment = align_left
            c.fill = fill_white
        elif c_idx in [1, 2, 4, 7, 8]:
            c.alignment = align_center
            c.fill = f_current
            if c_idx in [1, 4]:
                c.font = font_bold
        else:
            c.alignment = align_left
            c.fill = f_current

# -------------------------------------------------------------------------
# Sheet 4: Keputusan_Pembersihan_Data
# -------------------------------------------------------------------------
ws4 = wb.create_sheet(title="Keputusan_Pembersihan_Data")
ws4.views.sheetView[0].showGridLines = True
ws4.freeze_panes = "A2"

headers_dec = [
    "No", "Tipe Keputusan", "Objek / Elemen Terkait", "Kondisi di AMS Legacy",
    "Standardisasi Target Smart", "Alasan Bisnis & Teknis (Rationale)", "Dampak Terhadap Aset"
]

for col_idx, h in enumerate(headers_dec, start=1):
    cell = ws4.cell(row=1, column=col_idx, value=h)
    cell.font = font_header
    cell.fill = fill_header
    cell.alignment = align_center
    cell.border = border_header

decisions_data = [
    (
        1, "Penggabungan Kategori", "PROP (Properti & Gedung)",
        "Terpisah sebagai 170000 PROPERTI dan 220000 GEDUNG",
        "Digabung menjadi satu kategori: Properti & Gedung (PROP, ID: 8)",
        "Gedung dan properti tanah/bangunan memiliki karakteristik perlakuan aset dan amortisasi yang identik dalam ERP.",
        "Aset gedung maupun tanah kini terkonsolidasi dalam satu klasifikasi induk."
    ),
    (
        2, "Penggabungan Kategori", "KEND (Kendaraan)",
        "Terpisah sebagai 200000 KENDARAAN dan 250000 OTOMOTIF",
        "Digabung menjadi satu kategori: Kendaraan (KEND, ID: 10)",
        "Redundansi semantik. Semua armada operasional dikelompokkan dalam satu kategori kendaraan.",
        "Mencegah kebingungan penginputan aset kendaraan dinas/operasional."
    ),
    (
        3, "Pembentukan Kategori Baru", "TLKM (Telekomunikasi)",
        "Tersebar di Aksesoris (110004 Telepon, 110030 LCEX), Peripheral (140007 Fax), dan Elektronik (240013 HT, 240015 Smart EX)",
        "Dibuatkan kategori mandiri: Telekomunikasi (TLKM, ID: 13)",
        "Perangkat komunikasi memiliki vendor, pemeliharaan berkala, dan penanggung jawab IT telekomunikasi yang spesifik.",
        "Seluruh perangkat komunikasi kini terkelola rapi di bawah kategori TLKM."
    ),
    (
        4, "Reklasifikasi & Resolusi Bentrok Kode", "ACC-WB & MES-WTH",
        "Whiteboard (110002) dan Water Heater (110022) sama-sama berada di parent ACC dengan kode singkatan 'WH'",
        "Whiteboard tetap di Aksesoris (ACC-WB) | Water Heater dipindahkan ke Mesin (MES-WTH)",
        "Water Heater merupakan peralatan mekanikal utilitas gedung, lebih tepat berada di bawah kategori Mesin (MES) sekaligus menuntaskan bentrok kode WH.",
        "Aset water heater terkelompok di bawah Mesin dengan penomoran unik MES-WTH tanpa bentrok."
    ),
    (
        5, "Deduplikasi Subkategori", "MES-LIF (Lift Gedung)",
        "Tercatat ganda: di bawah Mesin (180005, LIF) dan Elektronik (240017, LT)",
        "Dikonolidasikan menjadi satu subkategori: Lift (MES-LIF, ID: 70)",
        "Lift adalah utilitas mesin gedung. Kedua kode legacy tetap dipetakan ke ID yang sama agar relasi data lama tidak putus.",
        "Data aset lift eksisting dari kedua kode AMS akan bergabung mulus ke MES-LIF."
    ),
    (
        6, "Deduplikasi Subkategori", "KEND-MO (Mobil)",
        "Tercatat ganda: 200001 (KENDARAAN - Operasional RE) dan 200002 (MOBIL)",
        "Dikonsolidasikan menjadi satu subkategori: Mobil (KEND-MO, ID: 74)",
        "Subkategori 200001 redundan dengan 200002. Keduanya merepresentasikan unit kendaraan mobil.",
        "Aset mobil lama terpetakan ke subkategori yang tepat."
    ),
    (
        7, "Penautan Subkategori Yatim", "24 Subkategori Relegated",
        "24 subkategori memiliki control='T' namun kolom parent bernilai NULL (misal AC Split, Genset, Panel, CCTV, TV)",
        "Dihubungkan ke kategori induk berdasarkan prefiks angka kode (26xxxx -> PDGR, 18xxxx -> MES, 24xxxx -> ELEK)",
        "Mencegah hilangnya subkategori penting akibat anomali relasi induk pada sistem AMS lama.",
        "Subkategori vital tetap tersedia di Smart dan dapat menampung ribuan aset terkait."
    ),
    (
        8, "Eksklusi Kategori Nonaktif", "PUR (210000 - Purnama)",
        "Kategori 210000 memiliki 0 subkategori di AMS",
        "Tidak dimasukkan ke daftar master kategori aktif Smart (tetap dicatat di staging audit)",
        "Kategori tidak memiliki definisi operasional dan tidak ada subkategori terdaftar.",
        "Menjaga kebersihan master data Smart agar hanya memuat kategori yang valid."
    )
]

for row_idx, rdata in enumerate(decisions_data, start=2):
    f_zebra = fill_zebra if row_idx % 2 == 1 else fill_white
    for col_idx, val in enumerate(rdata, start=1):
        cell = ws4.cell(row=row_idx, column=col_idx, value=val)
        cell.font = font_bold if col_idx in [1, 2, 3] else font_data
        cell.border = border_cell
        cell.fill = f_zebra
        if col_idx in [1, 2]:
            cell.alignment = align_center
        elif col_idx in [3, 4, 5]:
            cell.alignment = align_left
        else:
            cell.alignment = align_wrap_left

# -------------------------------------------------------------------------
# Sheet 5: Pemetaan_AMS_ke_Smart
# -------------------------------------------------------------------------
ws5 = wb.create_sheet(title="Pemetaan_AMS_ke_Smart")
ws5.views.sheetView[0].showGridLines = True
ws5.freeze_panes = "A2"

headers_map = [
    "Kode AMS", "ID AMS", "Tipe Objek Legacy", "Nama di AMS Legacy", "Deskripsi Legacy",
    "ID Kategori Smart", "Kode Kategori Smart", "ID Subkategori Smart", "Kode Subkategori Smart",
    "Nama Standar di Smart", "Catatan Relasi / Lineage"
]

for col_idx, h in enumerate(headers_map, start=1):
    cell = ws5.cell(row=1, column=col_idx, value=h)
    cell.font = font_header
    cell.fill = fill_header_sub
    cell.alignment = align_center
    cell.border = border_header

raw_sorted = raw.sort_values(by=["control", "code"]).reset_index(drop=True)
sub_map_dict = sub_map.set_index("ams_code").to_dict(orient="index")
cat_map_dict = cat_map.set_index("ams_code").to_dict(orient="index")

for row_idx, r in raw_sorted.iterrows():
    a_code = int(r["code"])
    a_id = int(r["id"])
    a_ctrl = r["control"]
    a_name = r["category"]
    a_desc = r["description"] if pd.notna(r["description"]) else "-"
    
    r_excel = row_idx + 2
    f_zebra = fill_zebra if r_excel % 2 == 1 else fill_white
    
    if a_ctrl == "F":
        obj_type = "Kategori Induk (F)"
        c_map = cat_map_dict.get(a_code, None)
        if c_map:
            c_smart_id = c_map["category_id"]
            c_smart_code = c_map["new_code"]
            c_smart_name = c_map["new_name"]
            lineage = "Dipetakan langsung ke Master Kategori"
        else:
            c_smart_id = "-"
            c_smart_code = "-"
            c_smart_name = "-"
            lineage = "Kategori nonaktif (Purnama) dieksklusi"
        s_smart_id = "-"
        s_smart_code = "-"
    else:
        obj_type = "Subkategori (T)"
        s_map = sub_map_dict.get(a_code, None)
        if s_map:
            c_smart_id = s_map["category_id"]
            c_smart_code = s_map["category_code"]
            s_smart_id = s_map["subcategory_id"]
            s_smart_code = s_map["subcategory_code"]
            c_smart_name = s_map["subcategory_name"]
            lineage = "Dipetakan ke Master Subkategori"
        else:
            c_smart_id = "-"
            c_smart_code = "-"
            s_smart_id = "-"
            s_smart_code = "-"
            c_smart_name = "-"
            lineage = "Anomali tidak terpetakan"
            
    row_vals = [
        a_code, a_id, obj_type, a_name, a_desc,
        c_smart_id, c_smart_code, s_smart_id, s_smart_code,
        c_smart_name, lineage
    ]
    
    for c_idx, val in enumerate(row_vals, start=1):
        c = ws5.cell(row=r_excel, column=c_idx, value=val)
        c.font = font_data
        c.border = border_cell
        c.fill = f_zebra
        if c_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = align_center
            if c_idx in [1, 7, 9]:
                c.font = font_bold
        else:
            c.alignment = align_left

# Validasi dropdown
dv = DataValidation(
    type="list",
    formula1='"Disetujui, Perlu Penyesuaian, Ditolak"',
    allow_blank=True,
    error="Nilai tidak valid. Pilih dari daftar: Disetujui, Perlu Penyesuaian, Ditolak.",
    errorTitle="Pilihan Tidak Valid",
    prompt="Silakan pilih status persetujuan",
    promptTitle="Status Persetujuan"
)
ws2.add_data_validation(dv)
dv.add(f"G2:G{len(cat_final) + 1}")

dv3 = DataValidation(
    type="list",
    formula1='"Disetujui, Perlu Penyesuaian, Ditolak"',
    allow_blank=True,
    error="Nilai tidak valid. Pilih dari daftar: Disetujui, Perlu Penyesuaian, Ditolak.",
    errorTitle="Pilihan Tidak Valid",
    prompt="Silakan pilih status persetujuan",
    promptTitle="Status Persetujuan"
)
ws3.add_data_validation(dv3)
dv3.add(f"K2:K{len(sub_final) + 1}")

# Lebar kolom otomatis
for ws in wb.worksheets:
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if ws.title == "Ringkasan_Eksekutif" and cell.row in [2, 3, 4, 7, 8, 9, 21, 22, 23, 24, 25, 26]:
                continue
            val_str = str(cell.value or "")
            if len(val_str) > max_len:
                max_len = len(val_str)
        ws.column_dimensions[col_letter].width = min(max(max_len + 4, 12), 48)

ws1.column_dimensions["A"].width = 4
ws1.column_dimensions["B"].width = 28
ws1.column_dimensions["C"].width = 24
ws1.column_dimensions["D"].width = 26
ws1.column_dimensions["E"].width = 45

ws2.column_dimensions["A"].width = 10
ws2.column_dimensions["B"].width = 15
ws2.column_dimensions["C"].width = 24
ws2.column_dimensions["D"].width = 24
ws2.column_dimensions["E"].width = 18
ws2.column_dimensions["F"].width = 50
ws2.column_dimensions["G"].width = 22
ws2.column_dimensions["H"].width = 35

ws3.column_dimensions["A"].width = 10
ws3.column_dimensions["B"].width = 14
ws3.column_dimensions["C"].width = 20
ws3.column_dimensions["D"].width = 18
ws3.column_dimensions["E"].width = 26
ws3.column_dimensions["F"].width = 32
ws3.column_dimensions["G"].width = 20
ws3.column_dimensions["H"].width = 18
ws3.column_dimensions["I"].width = 24
ws3.column_dimensions["J"].width = 36
ws3.column_dimensions["K"].width = 22
ws3.column_dimensions["L"].width = 35

wb.save(EXCEL_PATH)
print(f"File Excel Review berhasil disimpan: {EXCEL_PATH.resolve()}")

# Buat Markdown
md_content = f"""# Ringkasan Eksekutif & Panduan Tinjauan Master Data (Kategori & Subkategori)
### Migrasi Data Legacy AMS ke Microsoft SQL Server Smart
**Tanggal Dokumen:** {datetime.now().strftime('%d %B %Y')}  
**Target Pembaca:** Tim Manajemen Aset & Tim IT  
**File Excel Review Utama:** [`Review_Master_Kategori_dan_Subkategori.xlsx`](Review_Master_Kategori_dan_Subkategori.xlsx)

---

## 1. Urgensi & Latar Belakang
Master data Kategori dan Subkategori merupakan fondasi relasional utama dalam sistem manajemen aset. Sebelum proses pembersihan ribuan unit aset fisik dijalankan, struktur master data harus disepakati dan memperoleh **persetujuan (approval)** dari Tim Pengelola Aset dan Tim IT.

Dokumen ini dan file Excel terlampir disusun untuk memudahkan kedua tim dalam melakukan review, memberikan masukan, dan menandatangani persetujuan secara formal.

---

## 2. Ringkasan Transformasi Data (AMS Legacy vs Smart)

| Indikator Transformasi | Kondisi AMS Legacy | Hasil Standar Smart | Catatan Transformasi |
|:---|:---:|:---:|:---|
| **Kategori Induk** | 15 Baris (`control='F'`) | **13 Kategori Aktif** | Penggabungan kategori duplikat (`PROP` & `KEND`), eksklusi kategori nonaktif (`PUR`). |
| **Subkategori** | 113 Baris (`control='T'`) | **110 Subkategori Standar** | Deduplikasi `LIFT` & `MOBIL`, penamaan terstandar `<INDUK>-<ANAK>`. |
| **Subkategori Yatim (Orphan)** | 24 Item (`parent = NULL`) | **0 Subkategori Yatim** | 24 item dihubungkan kembali ke kategori induk sesuai prefiks kode legacy. |
| **Kategori Baru Mandiri** | Belum terpisah | **1 Kategori (`TLKM`)** | Perangkat telekomunikasi (Telepon, Smartphone, HT, Fax) dipisah dari Aksesoris/Elektronik. |
| **Resolusi Bentrok Kode & Reklasifikasi** | 1 Bentrok (`WH`) | **Terselesaikan** | `ACC-WB` (Whiteboard tetap di Aksesoris) & `MES-WTH` (Water Heater dipindah ke Mesin). |

---

## 3. Daftar Master Kategori Smart (13 Kategori)

| ID | Kode | Nama Kategori | Kode AMS Asal | Subkategori | Deskripsi / Cakupan Operasional |
|:--:|:----:|:--------------|:-------------:|:-----------:|:--------------------------------|
| 1 | `FUR` | Furniture | 100000 | 8 | Perabotan kantor (kursi, meja, lemari, rak, sofa). |
| 2 | `ACC` | Aksesoris | 110000 | 40 | Aksesoris & fixture kantor (whiteboard, jam dinding, bracket, brankas). |
| 3 | `COMP` | Computer | 120000 | 4 | Komputer desktop, notebook, workstation, server IT. |
| 4 | `MON` | Monitor | 130000 | 2 | Monitor komputer (tabung, LCD/LED display). |
| 5 | `PERP` | Peripheral | 140000 | 14 | Perangkat periferal, storage eksternal, multimedia, networking. |
| 6 | `PRNT` | Printer | 150000 | 6 | Printer inkjet, laserjet, plotter, ID card printer. |
| 7 | `SOFT` | Software | 160000 | 2 | Lisensi perangkat lunak engineering dan aplikasi perkantoran. |
| 8 | `PROP` | Properti & Gedung | 170000, 220000 | 1 | Konsolidasi properti tanah dan bangunan gedung. |
| 9 | `MES` | Mesin | 180000 | 5 | Mesin dan utilitas gedung (water heater, genset, lift, panel listrik, pompa air). |
| 10 | `KEND` | Kendaraan | 200000, 250000 | 1 | Konsolidasi kendaraan dinas/operasional dan otomotif. |
| 11 | `ELEK` | Elektronik | 240000 | 18 | Peralatan elektronik umum, keamanan gedung, sensor, sound system (CCTV, TV). |
| 12 | `PDGR` | Pendingin Ruangan | 260000 | 5 | Sistem tata udara dan pendingin ruangan (AC Split, AC Portable, AHU). |
| 13 | `TLKM` | Telekomunikasi | 110004, 140007, 240013, 240015, 110030 | 4 | Kategori mandiri baru untuk perangkat telekomunikasi (Telepon, Smartphone, HT, Fax). |

---

## 4. Keputusan Penting Pembersihan Data (Changelog & Rationale)

1. **Konsolidasi Properti & Gedung (`PROP`)**:
   - *Kondisi Lama*: Terpisah menjadi `170000 PROPERTI` dan `220000 GEDUNG`.
   - *Keputusan*: Digabung menjadi **`PROP` (Properti & Gedung)** karena perlakuan aset dan amortisasi tanah/bangunan terpadu.
2. **Konsolidasi Kendaraan & Otomotif (`KEND`)**:
   - *Kondisi Lama*: Terpisah menjadi `200000 KENDARAAN` dan `250000 OTOMOTIF`.
   - *Keputusan*: Digabung menjadi **`KEND` (Kendaraan)** untuk menghilangkan kerancuan pengelompokan kendaraan.
3. **Pembentukan Kategori Telekomunikasi (`TLKM`)**:
   - *Kondisi Lama*: Tersebar di bawah Aksesoris, Peripheral, dan Elektronik.
   - *Keputusan*: Dibentuk kategori khusus **`TLKM`** agar tata kelola perangkat komunikasi kantor terstandar.
4. **Reklasifikasi Water Heater & Resolusi Bentrok Kode**:
   - *Kondisi Lama*: `Whiteboard` dan `Water Heater` berada di bawah Aksesoris (`ACC`) dan sama-sama memakai kode `WH`.
   - *Keputusan*: `Water Heater` dipindahkan ke kategori **Mesin (`MES`)** sebagai **`MES-WTH`** (karena merupakan peralatan utilitas mekanikal gedung), sedangkan `Whiteboard` tetap di **Aksesoris (`ACC`)** sebagai **`ACC-WB`**.
5. **Deduplikasi `LIFT`**:
   - *Kondisi Lama*: Muncul ganda di Mesin (`180005`, `LIF`) dan Elektronik (`240017`, `LT`).
   - *Keputusan*: Digabung menjadi **`MES-LIF`**. Kedua kode lama tetap dipetakan ke target yang sama di tabel lookup.
6. **Deduplikasi `MOBIL`**:
   - *Kondisi Lama*: Tercatat ganda sebagai `200001` (Kendaraan Operasional RE) dan `200002` (Mobil).
   - *Keputusan*: Dikonsolidasi ke **`KEND-MO`** (`Mobil`).
7. **Penautan Kembali 24 Subkategori Yatim**:
   - *Kondisi Lama*: 24 item berstatus subkategori (`control='T'`) memiliki `parent = NULL`.
   - *Keputusan*: Dipetakan ulang ke induk aslinya (`PDGR`, `MES`, `ELEK`) berdasarkan prefiks angka kodenya.
8. **Eksklusi Kategori Tidak Aktif (`210000 PURNAMA`)**:
   - *Kondisi Lama*: Tercatat di AMS dengan 0 subkategori.
   - *Keputusan*: Dikecualikan dari master aktif Smart, namun tetap diaudit di staging.

---

## 5. Panduan Pengisian Lembar Kerja Excel
Stakeholder diharapkan membuka file **`Review_Master_Kategori_dan_Subkategori.xlsx`**:
1. Masuk ke sheet **`Master_Kategori`** dan **`Master_Subkategori`**.
2. Isikan kolom **`Status Persetujuan`** (pilih melalui dropdown: *Disetujui*, *Perlu Penyesuaian*, atau *Ditolak*).
3. Apabila ada masukan atau kebutuhan perubahan nama/kategori, tuliskan pada kolom **`Catatan & Masukan Tim Aset & IT`**.
4. File yang telah diisi dapat dikembalikan kepada tim data migration untuk difinalisasi.
"""

with open(MD_PATH, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"File Markdown Ringkasan berhasil disimpan: {MD_PATH.resolve()}")


Data Dir: /home/jovyan/data
Output Excel: /home/jovyan/data/smart/review/Review_Master_Kategori_dan_Subkategori.xlsx
Loaded: Raw=128, Categories=13, Subcategories=110
File Excel Review berhasil disimpan: /home/jovyan/data/smart/review/Review_Master_Kategori_dan_Subkategori.xlsx
File Markdown Ringkasan berhasil disimpan: /home/jovyan/data/smart/review/ringkasan_review_kategori.md


## 2. Verifikasi Hasil Output Workbook
Memeriksa lembar kerja dan ukuran file hasil ekspor.

In [2]:
import openpyxl
from pathlib import Path

wb_check = openpyxl.load_workbook(EXCEL_PATH)
print("Lembar kerja yang terbentuk dalam file Excel:")
for s in wb_check.sheetnames:
    sheet = wb_check[s]
    print(f" - {s:<28} : {sheet.max_row} baris, {sheet.max_column} kolom")

print(f"\nUkuran file Excel: {EXCEL_PATH.stat().st_size / 1024:.2f} KB")
print(f"Ukuran file Markdown: {MD_PATH.stat().st_size / 1024:.2f} KB")
print("\nSeluruh proses ekspor review selesai dengan sukses!")


Lembar kerja yang terbentuk dalam file Excel:
 - Ringkasan_Eksekutif          : 26 baris, 8 kolom
 - Master_Kategori              : 14 baris, 8 kolom
 - Master_Subkategori           : 111 baris, 12 kolom
 - Keputusan_Pembersihan_Data   : 9 baris, 7 kolom
 - Pemetaan_AMS_ke_Smart        : 129 baris, 11 kolom

Ukuran file Excel: 29.70 KB
Ukuran file Markdown: 5.98 KB

Seluruh proses ekspor review selesai dengan sukses!
